# Impedance ML Engineering — Session 04
## Refresh statistical modeling and uncertainty
**Core: 60 minutes · Statistics · Optional extension: 20–40 minutes**

## Why this matters
How is baseline telemetry associated with age and array type, and how uncertain are those associations?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Define an estimand using one baseline visit per training patient.
- Fit OLS and report effect sizes and confidence intervals.
- Inspect residuals, leverage, and robust-SE sensitivity.
- Use patient resampling to revisit sampling variability.

## R-to-Python / prerequisite bridge
`lm(y~x+factor(g))` → `smf.ols("y ~ x + C(g)",data=d).fit()`; `confint` → `.conf_int()`; inference and predictive risk answer different questions.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
An **estimand** is the quantity your question asks for, such as an adjusted baseline association or an unadjusted group mean difference. OLS models the conditional mean as a linear function of specified terms. A coefficient is an association conditional on those terms; unmeasured confounding can prevent causal interpretation. Residual assumptions affect standard errors and predictive intervals.

A confidence interval concerns repeated-sampling uncertainty for an estimated quantity; an individual prediction interval also includes residual outcome variation. HC3 allows a heteroskedasticity-robust covariance estimate without changing OLS coefficients. A bootstrap approximates sampling variability by repeated resampling of the independent units. Its validity depends on the sampling structure, not merely on repeating a computation many times.

Imports: `import statsmodels.formula.api as smf`, `import statsmodels.api as sm`. Model accessors: `.params`, `.conf_int()`, `.resid`, `.fittedvalues`, `.get_influence()`, `.get_prediction(new_frame).summary_frame()`.

## Task 1 — Define the population and sample (5 min)
Load `visit_features.csv`, retaining only training-patient IDs from the fixed partition and window 0. Keep complete outcome/age/array/protocol rows. Count exclusions explicitly. Predict why the usable sample is below or equal to 54, not 360. Estimand: adjusted baseline association, not a causal treatment effect.

## Task 2 — Inspect distributions and confounding (7 min)
Plot baseline diagonal mean versus age, colored by array, and grouped distributions by array/protocol. Report group sizes and mean/SD or median/IQR as justified. Inspect age balance across array groups. Choose one association hypothesis before looking at model p-values.

## Task 3 — Fit the prespecified OLS model (8 min)
Import `statsmodels.formula.api as smf`. Center age at 50 and fit `diag_mean_kohm ~ age_centered + C(array_type) + C(protocol)`. Inspect `.params`, `.conf_int()`, `.nobs`. Interpret intercept and one coefficient in kΩ, including the category reference. Hint: formula encoding handles categorical reference levels; inspect parameter names rather than assuming order.

## Task 4 — Diagnose fit assumptions (8 min)
Plot residual versus fitted, residual versus age, and a normal Q–Q plot; inspect leverage and Cook’s distance through `.get_influence()`. Identify one potential issue and explain what it affects. Do not delete a point solely to obtain significance. Report a diagnostic even if it looks acceptable.

## Task 5 — Compare ordinary and HC3 uncertainty (6 min)
Use `.get_robustcov_results(cov_type="HC3")` on the same fitted model. Compare coefficient estimates and intervals. Acceptance: point estimates remain unchanged; standard errors may differ. Explain why robust SEs do not repair an incorrect mean relationship or confounding.

## Task 6 — Bootstrap a descriptive contrast (8 min)
For complete baseline rows, resample patients/rows with replacement using seed 42 for 500 repetitions and recompute array_B minus array_A mean. Keep duplicate sampled people as repeated draws. Report percentile interval and histogram; note resamples lacking a group. Do not confuse this unadjusted contrast with the adjusted OLS coefficient.

## Task 7 — Plot adjusted mean predictions (6 min)
Create an age grid over the observed training range, crossing array types and fixing protocol to a named reference. Use `.get_prediction(grid).summary_frame()` and display fitted means plus mean confidence bands. Contrast `mean_ci_*` with `obs_ci_*`: uncertainty about a mean differs from variation of an individual observation.

## Task 8 — Write the statistical result (4 min)
Report sample size, effect estimate with interval, one diagnostic, sensitivity result, and a limitation. Explain which conclusions would need experimental design or additional covariates. Suggest how this model can inform features without treating a p-value as a feature-selection algorithm.

## Retrieval practice
Calculate an age-group descriptive difference without modeling. State why dichotomizing age discards information and can change the estimand.

## Professional reasoning
1. What could go wrong if a post-hoc hypothesis is presented as prespecified?
2. Why do 144 cells per matrix not multiply the independent sample size?
3. What is the difference between an association interval and validation MAE?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Compare a centered quadratic age term on the exact same rows, inspect diagnostics, and label this exploratory. Review multiplicity if many associations are screened.

## API references
[statsmodels OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html) · [mixed-effects models](https://www.statsmodels.org/stable/mixed_linear.html)